# 네이버증권 API 데이터 수집

이 튜토리얼에서는 **Selenium**을 사용하여 네이버증권 웹사이트에서 주가 데이터를 수집하는 방법에 대해 학습합니다.

## 1. Selenium이란?
Selenium은 웹 브라우저를 자동화하는 도구입니다. 실제 브라우저를 제어하여 사용자가 하는 모든 동작(클릭, 입력, 스크롤 등)을 프로그래밍으로 수행할 수 있습니다.

### Selenium의 주요 용도
- 웹 스크래핑 (동적 웹페이지 데이터 수집)
- 웹 테스트 자동화
- 반복적인 웹 작업 자동화

---

## 2. 필요한 라이브러리 설치
먼저 Selenium과 Chrome WebDriver를 설치합니다.

In [1]:
# Selenium 설치
# !pip install selenium

# Chrome WebDriver 다운로드 (사용하는 Chrome 버전에 맞는 드라이버 필요)
# https://sites.google.com/chromium.org/driver/ 에서 다운로드

---

## 3. 기본 설정
Selenium을 사용하여 네이버증권에 접속하고 HMM의 주가 데이터를 수집하는 예제입니다.

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import time

### Chrome WebDriver 설정

In [3]:
# Headless 모드로 Chrome 실행 (브라우저 창을 표시하지 않음)
options = Options()
options.add_argument('--headless')  # 브라우저 창 숨기기
options.add_argument('--no-sandbox')  # 리눅스 환경에서 필수
options.add_argument('--disable-dev-shm-usage')  # 메모리 문제 방지
options.add_argument('--disable-gpu')  # GPU 비활성화
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')

# WebDriver 실행 (chromedriver 경로 지정 필요)
# driver = webdriver.Chrome(options=options)

---

## 4. 네이버증권에서 데이터 수집하기

In [4]:
# 네이버증권 HMM 종목 페이지 접속
stock_code = '000660'  # HMM
url = f'https://m.stock.naver.com/domestic/stock/{stock_code}/total'

# driver.get(url)
# time.sleep(3)  # 페이지 로딩 대기

# 페이지 소스 가져오기
# html = driver.page_source
# soup = BeautifulSoup(html, 'html.parser')

---

## 5. requests를 사용한 API 데이터 수집
네이버증권은 API를 통해 데이터를 제공합니다. requests 라이브러리를 사용하면 간단하게 데이터를 가져올 수 있습니다.

In [5]:
import requests
import json

# 요청 헤더 설정
headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
    'origin': 'https://m.stock.naver.com',
    'referer': 'https://m.stock.naver.com/',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

# API 요청 파라미터
params = {
    'periodType': 'dayCandle',
}

# HMM(000660)의 일별 캔들 데이터 가져오기
stock_code = '000660'  # HMM
url = f'https://api.stock.naver.com/chart/domestic/item/{stock_code}'

response = requests.get(url, params=params, headers=headers)
data = response.json()

# JSON 파일로 저장
with open('네이버증권_데이터.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print('데이터가 네이버증권_데이터.json 파일로 저장되었습니다.')
print(data)

데이터가 네이버증권_데이터.json 파일로 저장되었습니다.
{'candles': [{'startPrice': 185100, 'closePrice': 185800, ...}]


---

## 6. 데이터 구조 확인
가져온 데이터의 구조를 확인하고 필요한 데이터만 추출합니다.

In [6]:
# 데이터 구조 확인
# candles 키 안에 일별 캔들 데이터가 있음
# print(data.keys())
# print(data['candles'][0])  # 첫 번째 데이터 확인

---

## 7. Selenium으로 더 많은 데이터 수집하기
Selenium을 사용하면 페이지 내의 모든 요소를 직접 조작할 수 있습니다.

In [7]:
# 전체 Selenium 스크래핑 예제
"""
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time

# Chrome 옵션 설정
options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# 드라이버 시작
driver = webdriver.Chrome(options=options)

# 페이지 접속
url = 'https://m.stock.naver.com/domestic/stock/000660/total'
driver.get(url)

# 페이지 로딩 대기
time.sleep(3)

# BeautifulSoup으로 파싱
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

# 데이터 추출 (실제 사이트 구조에 맞게 선택자 변경 필요)
# prices = soup.select('.stock_price')
# for price in prices:
#     print(price.text)

# 드라이버 종료
driver.quit()
"""

---

## 8. 정리 및 주의사항

### 주의사항
1. **robots.txt 확인**: 웹사이트의 이용약관과 robots.txt를 확인하세요.
2. **요청 간격**: 서버에 과부하를 주지 않도록适当的한 간격을 두세요.
3. **동의 없이 데이터 수집**: 상업적 목적으로 사용 시 법적 문제가 발생할 수 있습니다.
4. **API利用規約**: 네이버증권 API를 사용하려면 관련 이용약관에 동의해야 합니다.

### Selenium vs requests
| 방법 | 장점 | 단점 |
|---|---|---|
| **requests** | 빠르고 가벼움 | 동적 웹페이지 불가 |
| **Selenium** | 동적 웹페이지 가능 | 느리고 리소스 많이 사용 |